In [3]:
!pip install langdetect

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 26.7 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993332 sha256=d7e75e042c40e912a76e7079f11fedcb3907bc6bb65eea0dc1fc6aae11f32be0
  Stored in directory: /home/vramon/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect
ERROR: Could not install packages due to an OSError: [Errno 13] Permission denied: '/opt/jupyterhub/venv/lib/python3.12/site-packages/langdetect'
Check the permissions.


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import re
import pandas as pd
from difflib import SequenceMatcher
from datetime import datetime
from pathlib import Path
from typing import Dict, Tuple, List, Optional

import torch

# Sentence embeddings
from sentence_transformers import SentenceTransformer

# Language detection (tries langid first, then langdetect)
try:
    import langid  # pip: langid
    _LANG_DETECT_BACKEND = "langid"
except Exception:
    langid = None
    _LANG_DETECT_BACKEND = None

try:
    from langdetect import detect as _ld_detect  # pip: langdetect
    _LANG_DETECT_BACKEND = _LANG_DETECT_BACKEND or "langdetect"
except Exception:
    _ld_detect = None
    _LANG_DETECT_BACKEND = _LANG_DETECT_BACKEND or None

# ----------------
# Paths
# ----------------
ENTITY_IN  = "entity_translations_ca.revoted.csv"
REL_IN     = "relation_translations_ca.revoted.csv"

ENTITY_OUT = "entity_translations_ca.revoted.csv"
REL_OUT    = "relation_translations_ca.revoted.csv"

VOTE_THRESHOLD_DEFAULT = 0.90

# ----------------
# Language filtering policy
# ----------------
TARGET_LANG = "ca"

# If True: keep ONLY candidates detected as Catalan
# If False: discard only candidates detected as English (keeps other non-English outputs)
STRICT_CA_ONLY = False

# Always discard if detected English (unless detector fails and returns None/"und")
DISCARD_ENGLISH = True

# ----------------
# Similarity + voting (string-based)
# ----------------
_ws_re = re.compile(r"\s+")

def _norm(s) -> str:
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    return _ws_re.sub(" ", str(s).strip())

def _sim(a: str, b: str) -> float:
    return SequenceMatcher(None, _norm(a).lower(), _norm(b).lower()).ratio()

MOJIBAKE_PATTERNS = ("Ã", "Â", "�")

def mojibake_penalty(text) -> float:
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return 0.0
    t = str(text)
    return 0.2 if any(p in t for p in MOJIBAKE_PATTERNS) else 0.0

def vote_details(t1: str, t2: str, t3: str, threshold: float):
    s12 = _sim(t1, t2) if _norm(t1) and _norm(t2) else 0.0
    s13 = _sim(t1, t3) if _norm(t1) and _norm(t3) else 0.0
    s23 = _sim(t2, t3) if _norm(t2) and _norm(t3) else 0.0
    return {
        "sim_nllb_madlad": float(f"{s12:.4f}"),
        "sim_nllb_salamandra": float(f"{s13:.4f}"),
        "sim_madlad_salamandra": float(f"{s23:.4f}"),
        "agree_nllb_madlad": int(s12 >= threshold),
        "agree_nllb_salamandra": int(s13 >= threshold),
        "agree_madlad_salamandra": int(s23 >= threshold),
    }

def safe_thr(x) -> float:
    try:
        v = float(x)
        if v <= 0 or v > 1.0:
            return VOTE_THRESHOLD_DEFAULT
        return v
    except Exception:
        return VOTE_THRESHOLD_DEFAULT

# ----------------
# Language detection
# ----------------
def detect_lang(text: str) -> str:
    """
    Returns ISO 639-1 code (e.g., 'ca', 'en', ...) or 'und' if unknown.
    """
    t = _norm(text)
    if not t:
        return "und"

    # langid
    if langid is not None:
        try:
            code, score = langid.classify(t)
            return code or "und"
        except Exception:
            pass

    # langdetect
    if _ld_detect is not None:
        try:
            return _ld_detect(t) or "und"
        except Exception:
            return "und"

    # No detector available
    return "und"

def keep_candidate(lang_code: str) -> bool:
    """
    Policy:
      - If STRICT_CA_ONLY: keep only 'ca'
      - Else: discard only English if DISCARD_ENGLISH
    """
    if STRICT_CA_ONLY:
        return lang_code == TARGET_LANG
    if DISCARD_ENGLISH and lang_code == "en":
        return False
    return True

# ----------------
# GTE embeddings + cosine similarity
# ----------------
GTE_MODEL_ID = "Alibaba-NLP/gte-multilingual-base"

def get_device() -> str:
    return "cuda" if torch.cuda.is_available() else "cpu"

gte = SentenceTransformer(GTE_MODEL_ID, device=get_device(), trust_remote_code=True)

def build_embedding_cache(texts: List[str], batch_size: int = 256) -> Dict[str, torch.Tensor]:
    uniq, seen = [], set()
    for t in texts:
        t = _norm(t)
        if not t or t in seen:
            continue
        seen.add(t)
        uniq.append(t)

    cache: Dict[str, torch.Tensor] = {}
    if not uniq:
        return cache

    embs = gte.encode(
        uniq,
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    embs = embs.detach().cpu()
    for t, e in zip(uniq, embs):
        cache[t] = e
    return cache

def cos_from_cache(src: str, hyp: str, cache: Dict[str, torch.Tensor]) -> Optional[float]:
    src_n = _norm(src)
    hyp_n = _norm(hyp)
    if not src_n or not hyp_n:
        return None
    e1 = cache.get(src_n)
    e2 = cache.get(hyp_n)
    if e1 is None or e2 is None:
        return None
    return float(torch.dot(e1, e2).item())  # normalized => cosine

# ----------------
# Vote using string agreement + GTE cosine as selector (with filtered candidates)
# ----------------
def vote_3way_with_gte_filtered(
    t1: str, t2: str, t3: str,
    cos1: Optional[float], cos2: Optional[float], cos3: Optional[float],
    thr: float = 0.90,
) -> Tuple[str, str]:
    """
    Returns (winner_text, winner_key). Assumes candidates already filtered (emptied if discarded).
    """
    texts = {
        "nllb": _norm(t1),
        "madlad": _norm(t2),
        "salamandra": _norm(t3),
    }
    # Only consider non-empty candidates
    cands = [k for k in ["nllb", "madlad", "salamandra"] if texts[k]]
    if not cands:
        return "", ""

    coss = {
        "nllb": (cos1 if cos1 is not None else float("-inf")),
        "madlad": (cos2 if cos2 is not None else float("-inf")),
        "salamandra": (cos3 if cos3 is not None else float("-inf")),
    }

    def s(a, b):
        if a == b:
            return 1.0
        if not texts[a] or not texts[b]:
            return 0.0
        return _sim(texts[a], texts[b])

    # Thresholded edges on remaining candidates
    edges = []
    for i in range(len(cands)):
        for j in range(i + 1, len(cands)):
            a, b = cands[i], cands[j]
            if s(a, b) >= thr:
                edges.append((a, b))

    adj = {c: set() for c in cands}
    for a, b in edges:
        adj[a].add(b)
        adj[b].add(a)

    seen = set()
    comps = []
    for c in cands:
        if c in seen:
            continue
        stack = [c]
        comp = set()
        while stack:
            x = stack.pop()
            if x in seen:
                continue
            seen.add(x)
            comp.add(x)
            stack.extend(adj[x])
        comps.append(comp)

    comps.sort(key=lambda comp: (-len(comp), sorted(comp)))
    best_comp = comps[0] if comps else set(cands)

    def medoid_score(k: str) -> float:
        # sum sims to all remaining candidates minus mojibake penalty
        sc = 0.0
        for other in cands:
            sc += s(k, other)
        sc -= 1.0  # remove self similarity
        return sc - mojibake_penalty(texts[k])

    def tie_score(k: str) -> Tuple[float, float]:
        return (coss[k], medoid_score(k))

    # If there's a size-2 agreement component (in remaining), choose within it by cosine→medoid
    if len(best_comp) == 2:
        pair = list(best_comp)
        winner_key = max(pair, key=lambda k: tie_score(k))
        return texts[winner_key], winner_key

    # Otherwise choose best cosine; tie-break medoid
    if all(coss[k] == float("-inf") for k in cands):
        winner_key = max(cands, key=lambda k: medoid_score(k))
        return texts[winner_key], winner_key

    winner_key = max(cands, key=lambda k: tie_score(k))
    return texts[winner_key], winner_key

# ----------------
# Per-file revoting
# ----------------
GTE_COLS = ["cos_en_nllb", "cos_en_madlad", "cos_en_salamandra", "gte_winner"]
LANG_COLS = ["lang_nllb", "lang_madlad", "lang_salamandra"]

def ensure_cols(df: pd.DataFrame, cols: List[str]):
    for c in cols:
        if c not in df.columns:
            df[c] = pd.NA

def revote_entities(df: pd.DataFrame, emb_cache: Dict[str, torch.Tensor]) -> pd.DataFrame:
    df = df.copy()

    ensure_cols(df, [
        "sim_nllb_madlad","sim_nllb_salamandra","sim_madlad_salamandra",
        "agree_nllb_madlad","agree_nllb_salamandra","agree_madlad_salamandra",
        "mt_voted","final_text","final_source","processed_at",
        *GTE_COLS, *LANG_COLS
    ])

    for i, row in df.iterrows():
        src_en = row.get("source_query_text", row.get("source_token", ""))

        t1_raw = row.get("mt_nllb", "")
        t2_raw = row.get("mt_madlad", "")
        t3_raw = row.get("mt_salamandra", "")
        thr = safe_thr(row.get("vote_threshold", VOTE_THRESHOLD_DEFAULT))

        # Detect languages
        l1 = detect_lang(t1_raw)
        l2 = detect_lang(t2_raw)
        l3 = detect_lang(t3_raw)
        df.at[i, "lang_nllb"] = l1
        df.at[i, "lang_madlad"] = l2
        df.at[i, "lang_salamandra"] = l3

        # Filter candidates by language policy
        t1 = t1_raw if keep_candidate(l1) else ""
        t2 = t2_raw if keep_candidate(l2) else ""
        t3 = t3_raw if keep_candidate(l3) else ""

        # recompute string sim/agree using filtered candidates
        if any(_norm(x) for x in (t1, t2, t3)):
            det = vote_details(t1, t2, t3, thr)
            for k, v in det.items():
                df.at[i, k] = v

        # compute GTE cosines vs EN source for filtered candidates
        cos1 = cos_from_cache(src_en, t1, emb_cache)
        cos2 = cos_from_cache(src_en, t2, emb_cache)
        cos3 = cos_from_cache(src_en, t3, emb_cache)
        df.at[i, "cos_en_nllb"] = cos1 if cos1 is not None else pd.NA
        df.at[i, "cos_en_madlad"] = cos2 if cos2 is not None else pd.NA
        df.at[i, "cos_en_salamandra"] = cos3 if cos3 is not None else pd.NA

        final_source = str(row.get("final_source", "") or "")
        if final_source.startswith("wikidata_") or final_source.startswith("passthrough_"):
            continue

        # Only revote if MT-based (or empty/unknown source but has candidates)
        if (final_source == "mt_vote") or (final_source == "" and any(_norm(x) for x in (t1, t2, t3))):
            voted, winner_key = vote_3way_with_gte_filtered(
                t1, t2, t3, cos1, cos2, cos3, thr=thr
            )
            # If everything got filtered out, keep previous final_text (do not blank it)
            if voted:
                df.at[i, "mt_voted"] = voted
                df.at[i, "final_text"] = voted
                df.at[i, "final_source"] = "mt_vote_revoted_gte_langfilter"
                df.at[i, "gte_winner"] = winner_key
                df.at[i, "processed_at"] = datetime.now().isoformat(timespec="seconds")

    return df

def revote_relations(df: pd.DataFrame, emb_cache: Dict[str, torch.Tensor]) -> pd.DataFrame:
    df = df.copy()

    ensure_cols(df, [
        "sim_nllb_madlad","sim_nllb_salamandra","sim_madlad_salamandra",
        "agree_nllb_madlad","agree_nllb_salamandra","agree_madlad_salamandra",
        "mt_voted","final_text","final_source","processed_at",
        *GTE_COLS, *LANG_COLS
    ])

    for i, row in df.iterrows():
        src_en = row.get("source_query_text", row.get("source_token", ""))

        t1_raw = row.get("mt_nllb", "")
        t2_raw = row.get("mt_madlad", "")
        t3_raw = row.get("mt_salamandra", "")
        thr = safe_thr(row.get("vote_threshold", VOTE_THRESHOLD_DEFAULT))

        # Detect languages
        l1 = detect_lang(t1_raw)
        l2 = detect_lang(t2_raw)
        l3 = detect_lang(t3_raw)
        df.at[i, "lang_nllb"] = l1
        df.at[i, "lang_madlad"] = l2
        df.at[i, "lang_salamandra"] = l3

        # Filter by language policy
        t1 = t1_raw if keep_candidate(l1) else ""
        t2 = t2_raw if keep_candidate(l2) else ""
        t3 = t3_raw if keep_candidate(l3) else ""

        det = vote_details(t1, t2, t3, thr)
        for k, v in det.items():
            df.at[i, k] = v

        cos1 = cos_from_cache(src_en, t1, emb_cache)
        cos2 = cos_from_cache(src_en, t2, emb_cache)
        cos3 = cos_from_cache(src_en, t3, emb_cache)
        df.at[i, "cos_en_nllb"] = cos1 if cos1 is not None else pd.NA
        df.at[i, "cos_en_madlad"] = cos2 if cos2 is not None else pd.NA
        df.at[i, "cos_en_salamandra"] = cos3 if cos3 is not None else pd.NA

        voted, winner_key = vote_3way_with_gte_filtered(
            t1, t2, t3, cos1, cos2, cos3, thr=thr
        )
        if voted:
            df.at[i, "mt_voted"] = voted
            df.at[i, "final_text"] = voted
            df.at[i, "final_source"] = "mt_vote_revoted_gte_langfilter"
            df.at[i, "gte_winner"] = winner_key
            df.at[i, "processed_at"] = datetime.now().isoformat(timespec="seconds")

    return df

# ----------------
# Run
# ----------------
entity_in_path = Path(ENTITY_IN)
rel_in_path = Path(REL_IN)

entities = pd.read_csv(entity_in_path)
relations = pd.read_csv(rel_in_path)

if "kind" in entities.columns:
    entities = entities[entities["kind"].astype(str) == "entity"].copy()
if "kind" in relations.columns:
    relations = relations[relations["kind"].astype(str) == "relation"].copy()

# Build embedding cache for all EN sources + all MT outputs
all_texts: List[str] = []

def add_texts_from_df(df: pd.DataFrame):
    if "source_query_text" in df.columns:
        all_texts.extend(df["source_query_text"].astype(str).tolist())
    if "source_token" in df.columns:
        all_texts.extend(df["source_token"].astype(str).tolist())
    for c in ["mt_nllb", "mt_madlad", "mt_salamandra"]:
        if c in df.columns:
            all_texts.extend(df[c].astype(str).tolist())

add_texts_from_df(entities)
add_texts_from_df(relations)

print(f"Language detector backend: {_LANG_DETECT_BACKEND or 'NONE'}")
print("Encoding embeddings for cosine similarity (gte-multilingual-base)...")
emb_cache = build_embedding_cache(all_texts, batch_size=256)

entities_rev = revote_entities(entities, emb_cache)
relations_rev = revote_relations(relations, emb_cache)

entities_rev.to_csv(ENTITY_OUT, index=False)
relations_rev.to_csv(REL_OUT, index=False)

print("Wrote:")
print("-", ENTITY_OUT)
print("-", REL_OUT)

if "final_text" in entities.columns:
    changed_entities = (entities_rev["final_text"].astype(str) != entities["final_text"].astype(str)).sum()
    print("Entity rows with changed final_text:", int(changed_entities))

if "final_text" in relations.columns:
    changed_rel = (relations_rev["final_text"].astype(str) != relations["final_text"].astype(str)).sum()
    print("Relation rows with changed final_text:", int(changed_rel))

Some weights of the model checkpoint at Alibaba-NLP/gte-multilingual-base were not used when initializing NewModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Language detector backend: langid
Encoding embeddings for cosine similarity (gte-multilingual-base)...


Batches:   0%|          | 0/45 [00:00<?, ?it/s]

Wrote:
- entity_translations_ca.revoted.csv
- relation_translations_ca.revoted.csv
Entity rows with changed final_text: 0
Relation rows with changed final_text: 0
